# 01 — Exploratory Data Analysis

Delivery ETA & Late-Risk dataset (45.6k orders). Quick structural pass over the **interim parquet** produced by the ingestion stage — the same artifact the training pipeline consumes, so no train/analysis skew.

**Run order**: `python scripts/fetch_data.py` → `python scripts/run_pipeline.py --stages ingest,validate` first, then run all cells.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import get, load_config

pd.set_option("display.width", 140)
cfg = load_config()
df = pd.read_parquet(get(cfg, "paths.interim_data"))
df.shape

## 1. Target: delivery time distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["time_taken_min"].hist(bins=50, ax=axes[0], color="tab:blue")
axes[0].set_title("Delivery time (min)")
axes[0].set_xlabel("minutes")
df["time_taken_min"].plot.box(ax=axes[1], vert=False)
axes[1].set_title("Boxplot")
plt.tight_layout()
plt.show()

df["time_taken_min"].describe().round(2)

## 2. Time structure: orders by date / hour

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["order_date"].value_counts().sort_index().plot(ax=axes[0], marker=".", lw=0)
axes[0].set_title("Orders per date")
axes[0].tick_params(axis="x", rotation=45)

hours = pd.to_datetime(df["time_ordered"], format="%H:%M:%S", errors="coerce").dt.hour
hours.value_counts().sort_index().plot(kind="bar", ax=axes[1], color="tab:blue")
axes[1].set_title("Orders by hour of day")
plt.tight_layout()
plt.show()

## 3. Categorical drivers: traffic, weather, city

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["road_traffic_density", "weather", "city"]):
    med = df.groupby(col)["time_taken_min"].median().sort_values()
    med.plot(kind="barh", ax=ax, color="tab:blue")
    ax.set_title(f"Median delivery time by {col}")
plt.tight_layout()
plt.show()

## 4. Geo: distance vs time

In [ ]:
from src.features.build_features import haversine_km

df["haversine_km"] = haversine_km(df["rest_lat"], df["rest_lon"], df["dest_lat"], df["dest_lon"])
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["haversine_km"], df["time_taken_min"], s=4, alpha=0.25, color="tab:blue")
ax.set_xlabel("Restaurant → drop haversine (km)")
ax.set_ylabel("Delivery time (min)")
ax.set_title("Distance vs delivery time")
plt.tight_layout()
plt.show()
df[["haversine_km", "time_taken_min"]].corr().iloc[0, 1].round(3)

## 5. Rider dimension

Rider-level medians are computed **train-only** in the pipeline (leak-safe target encoding). Here we just eyeball the raw relationship.

In [ ]:
rider_stats = df.groupby("rider_id")["time_taken_min"].agg(["median", "count", "std"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(rider_stats["median"], bins=40, color="tab:blue")
axes[0].set_title("Distribution of per-rider median delivery time")
axes[1].scatter(rider_stats["count"], rider_stats["std"], s=6, alpha=0.5)
axes[1].set_xlabel("orders (rider)")
axes[1].set_ylabel("std (rider)")
axes[1].set_title("Rider volume vs variability")
plt.tight_layout()
plt.show()

## 6. Missing-data map

Which columns arrive incomplete (handled by validation gates + fallbacks downstream).

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss[miss > 0].plot(kind="barh", figsize=(7, 4), color="tab:orange")
plt.title("Null share per column")
plt.xlabel("share")
plt.show()
miss[miss > 0].round(4)

## Summary

- Target is right-skewed with a long tail — motivates **quantile regression** over a single point estimate.
- Traffic density, weather and city shift the median delivery time materially — prime categorical drivers.
- Distance is only weakly correlated with time — the rider/temporal dimensions carry real signal.
- Several columns arrive with nulls — the model contract's fallback rules exist for exactly this.